# From Combine Speed Profiles to NFL Outcomes

This notebook builds prospect-level movement profiles and tests a focused hypothesis: does first-second acceleration in the Combine 40 add information about defensive linemen's rookie regular-season high-speed capacity beyond official tests and size available before the draft? It compares early acceleration with later phases, checks repeated-attempt reliability, audits raw 40-yard traces, propagates player-specific game-resampling uncertainty, and tests fixed-window and play-context-adjusted speed outcomes. The 2025 cohort is the only forward-in-time holdout; earlier leave-one-cohort-out folds also train on later classes.

The full-play peak-speed association is exploratory and does not replicate consistently in fixed post-snap windows, the context-adjusted four-second model, or pressure outcomes. It is not a draft grade, causal claim, or validated training prescription. The full 510-player profile explorer provides descriptive context; model selection and sample limits are reported alongside the lead.

**Competition input:** the official ZIP or extracted competition CSV directory. **Output:** player profiles, drill summaries, reports, and a self-contained prospect explorer in `outputs/`.

## Setup and data discovery

Attach the official competition input and the code-bundle ZIP. This cell locates either the mounted CSV directory or ZIP, safely extracts the source bundle, and sets shared paths. Run the notebook in order; intermediate results are saved under `outputs/`. Each script reports elapsed time so a long stage is visible in Kaggle logs.


In [ ]:
from pathlib import Path
import os, shutil, stat, subprocess, sys, time, zipfile
from pathlib import PurePosixPath, PureWindowsPath

INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
configured = os.environ.get('NFL_BDB_INPUT')
if configured:
    DATA_INPUT = Path(configured)
else:
    archives = list(INPUT_ROOT.rglob('nfl-big-data-bowl-2027.zip'))
    folders = [p.parent for p in INPUT_ROOT.rglob('combine_tracking.csv')]
    if len(archives) > 1 or (not archives and len(folders) > 1):
        raise RuntimeError('Multiple competition inputs found. Set NFL_BDB_INPUT to the exact ZIP or CSV directory.')
    DATA_INPUT = archives[0] if len(archives) == 1 else (folders[0] if len(folders) == 1 else None)
if DATA_INPUT is None or not DATA_INPUT.exists():
    raise FileNotFoundError('Attach the NFL Big Data Bowl input or set NFL_BDB_INPUT to its ZIP/CSV directory.')
os.environ['NFL_BDB_INPUT'] = str(DATA_INPUT)

PROJECT = Path.cwd()
if not (PROJECT / 'run_analysis.py').exists():
    code_roots = [p.parent for p in INPUT_ROOT.rglob('run_analysis.py')]
    bundles = list(INPUT_ROOT.rglob('nfl_speed_project_code.zip'))
    if len(code_roots) > 1 or (not code_roots and len(bundles) > 1):
        raise RuntimeError('Multiple code sources found. Attach one project source or configure it explicitly.')
    if len(code_roots) == 1:
        PROJECT = code_roots[0]
    elif len(bundles) == 1:
        PROJECT = Path('/kaggle/working/nfl_speed_project') if Path('/kaggle/working').exists() else Path.cwd() / 'nfl_speed_project'
        PROJECT.mkdir(parents=True, exist_ok=True)
        destination = PROJECT.resolve()
        with zipfile.ZipFile(bundles[0]) as zf:
            infos = zf.infolist()
            files, directories, seen = set(), set(), set()
            if len(infos) > 500:
                raise ValueError('Code archive exceeds the 500-member safety limit.')
            if sum(info.file_size for info in infos) > 128 * 1024 * 1024:
                raise ValueError('Code archive exceeds the 128 MiB uncompressed-size limit.')
            for info in infos:
                posix, windows = PurePosixPath(info.filename), PureWindowsPath(info.filename)
                if posix.is_absolute() or windows.is_absolute() or windows.drive or '..' in posix.parts or '\\' in info.filename:
                    raise ValueError(f'Unsafe code archive path: {info.filename!r}')
                normalized = posix.as_posix().rstrip('/')
                if not normalized or normalized in seen:
                    raise ValueError(f'Duplicate or empty code archive path: {info.filename!r}')
                seen.add(normalized)
                if not info.is_dir():
                    if info.file_size > 32 * 1024 * 1024:
                        raise ValueError(f'Code archive member exceeds the 32 MiB limit: {info.filename!r}')
                    files.add(normalized)
                else:
                    directories.add(normalized)
            for name in files:
                if name in directories or any(parent.as_posix() in files for parent in PurePosixPath(name).parents if parent.as_posix() != '.'):
                    raise ValueError(f'Code archive file/directory conflict: {name!r}')
                if any(directory.startswith(name + '/') for directory in directories):
                    raise ValueError(f'Code archive file/directory conflict: {name!r}')
            for info in infos:
                posix, windows = PurePosixPath(info.filename), PureWindowsPath(info.filename)
                target = (destination / Path(*posix.parts)).resolve()
                if target != destination and destination not in target.parents:
                    raise ValueError(f'Code archive path escapes destination: {info.filename!r}')
                if stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError(f'Symlinks are not allowed: {info.filename!r}')
                if info.is_dir(): target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(info) as src, target.open('wb') as dst: shutil.copyfileobj(src, dst)
    else:
        raise FileNotFoundError('Attach the project code ZIP as a Kaggle input, or run from the project directory.')

OUTPUT = PROJECT / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)

def run(script, *args):
    path = PROJECT / script
    if not path.exists(): raise FileNotFoundError(path)
    started = time.perf_counter()
    print(f'\n[RUN] {script}', flush=True)
    try:
        subprocess.run([sys.executable, str(path), *map(str, args)], check=True)
    except Exception:
        print(f'[FAILED] {script} after {time.perf_counter() - started:.1f}s', flush=True)
        raise
    elapsed = time.perf_counter() - started
    RUN_TIMINGS.append((script, elapsed))
    print(f'[DONE] {script} in {elapsed:.1f}s', flush=True)

RUN_TIMINGS = []



## Stage 1 — Build player profiles and audit the Combine traces

This stage derives movement profiles, checks data coverage, and audits the 40-yard measurement axis and processing choices before modeling.


In [ ]:
# Rebuild the profile and athlete-by-drill feature tables from source traces.
run('run_analysis.py', '--archive', DATA_INPUT, '--output', OUTPUT)
run('write_core_run_manifest.py', '--archive', DATA_INPUT, '--output', OUTPUT)
run('build_speed_signatures.py')
run('audit_project_data.py', '--archive', DATA_INPUT)


## Stage 2 — Test the speed-profile hypothesis

This stage checks attempt reliability, career-role and position alternatives, then evaluates the defensive-line early-acceleration signal across exposure thresholds and audits the game-tracking linkage.


In [ ]:
# Measurement reliability and held-out career-role transfer analyses.
run('analyze_40_repeatability.py')
run('analyze_40_feature_sensitivity.py', '--archive', DATA_INPUT)
run('audit_dl_early_accel_measurement_transfer.py')
run('audit_40_raw_trace_review.py', '--archive', DATA_INPUT)
run('audit_40_forward_axis_sensitivity.py', '--archive', DATA_INPUT)
run('model_career_speed_transfer.py', '--archive', DATA_INPUT)
run('model_position_specific_career.py')
run('model_db_tackle_transfer.py', '--archive', DATA_INPUT, '--min-opportunities', 50)
run('model_db_tackle_transfer.py', '--archive', DATA_INPUT, '--min-opportunities', 100)


## Stage 3 — Fit the defensive-line speed-transfer models

Run the position-specific early-acceleration models across exposure thresholds and check player-to-game linkage before generating any football-outcome claims.


In [ ]:
# Focused defensive-line transfer models and position-specific comparisons.
run('analyze_40_repeatability_by_position.py')
run('model_dl_early_accel_transfer.py', '--min-game-plays', 50)
run('model_dl_early_accel_transfer.py', '--min-game-plays', 100)
run('audit_dl_nested_phase_selection.py', '--min-game-plays', 100)
run('model_dl_early_accel_predraft.py', '--min-game-plays', 25)
run('model_dl_early_accel_predraft.py', '--min-game-plays', 50)
run('model_dl_early_accel_predraft.py', '--min-game-plays', 100)
# Check whether the signal depends on averaging two tracked 40-yard attempts.
run('audit_dl_attempt_specific_transfer.py')
run('model_dl_early_accel_predraft.py', '--min-game-plays', 200)
run('audit_game_tracking_linkage.py', '--archive', DATA_INPUT)


## Stage 4 — Build game context and test football outcomes

Construct blitz-opportunity and drill-derived features first, then evaluate burst, pressure, and game-speed targets. These are diagnostics of football translation, not proof of pass-rush value.


In [ ]:
# Build the Combine hoop features and rookie blitz-opportunity table first.
run('analyze_dl_hoop.py', '--archive', DATA_INPUT)
# These football-specific burst outcomes consume the opportunity table above.
run('analyze_dl_game_burst.py', '--archive', DATA_INPUT)
run('model_dl_transfer.py')
run('model_dl_pressure_context.py')
run('model_dl_40_accel_to_football_outcomes.py', '--min-burst-plays', 50)
run('model_dl_40_accel_to_football_outcomes.py', '--min-burst-plays', 100)
run('model_dl_game_speed_ablation.py', '--min-game-plays', 100)
run('audit_dl_early_accel_oof.py')
run('model_dl_game_speed_role_sensitivity.py', '--min-game-plays', 100)
run('model_game_speed_by_position.py', '--min-game-plays', 100)
run('model_position_specific_game_speed.py', '--min-game-plays', 100)
run('audit_position_phase_nested_selection.py')
run('build_position_phase_nested_figure.py')


## Stage 5 — Challenge the target and make the deliverables

Compare full-play peaks with fixed post-snap windows and matched play context, propagate game-sampling uncertainty, and run nested-selection and disjoint-game checks. Then generate the submission figures and player-profile explorer.


In [ ]:
# This extra pass re-reads game tracking to quantify p90 target uncertainty by game.
run('analyze_dl_game_speed_target_precision.py', '--archive', DATA_INPUT, '--draws', 1000)
run('build_dl_split_half_figure.py')
# Check whether the Combine signal survives fixed 1–4 second post-snap speed windows.
run('analyze_dl_fixed_window_target.py', '--archive', DATA_INPUT, '--min-plays', 50, 100)
# Evaluate four-second speed within matched play situations with equal player weighting.
run('analyze_dl_contextual_game_speed.py', '--archive', DATA_INPUT, '--min-plays', 50, 100)
run('model_dl_target_uncertainty.py', '--draws', 1000)
run('build_dl_transfer_figure.py')
run('build_dl_speed_profile_case_illustration.py')
run('build_profile_explorer.py')

print(f'\nCompleted {len(RUN_TIMINGS)} scripts in {sum(t for _, t in RUN_TIMINGS) / 60:.1f} minutes.')


## Main results and interpretation

The report below is regenerated from the competition files. The primary outcome is player-level rookie-season p90 per-play peak speed; plays estimate a player outcome, while the validation unit remains the player. Every model holds out a full draft cohort and fits preprocessing on training players only; only the 2025 fold is forward in time. Fixed-window tests use p90 peak speed during 1–4-second windows after ball snap. Pooled intervals cross zero; forward-2025 effects vary by horizon and alpha, including worse 1-second alpha-5 error. The early-only model has 46 DLs at the 100-play threshold; the all-phase ablation has 34 complete DLs. A nested audit also selects among 12 position-by-phase candidates inside outer cohort holdouts on 161 complete players. Player-bootstrap intervals use fixed OOF predictions. The separate game-resampling simulation independently draws each player's outcome and refits models; it omits shared game, opponent, and weather shocks. A split-half audit partitions each eligible player's rookie games into disjoint, play-count-balanced halves, quantifies p90 rank repeatability, and refits the same model on each half-specific outcome; its partition range is not a confidence interval or independent validation.

The career outcome is a participation/role proxy, not pure player ability: injury, team depth charts, opportunity, and scheme affect snaps. A null increment means this feature block has not shown added predictive value for this outcome in these cohorts; it does not prove that Combine tracking is useless for every position or role.

In [ ]:
from IPython.display import display, SVG, HTML

for name in ['combine_40_repeatability_report.md', 'forty_feature_sensitivity_report.md', 'forty_raw_trace_review_report.md', 'forty_forward_axis_sensitivity_report.md', 'dl_nested_phase_selection_report_100plays.md', 'career_speed_transfer_report.md', 'position_specific_career_report.md', 'db_tackle_transfer_report_50plus.md', 'db_tackle_transfer_report_100plus.md', 'forty_repeatability_by_position_report.md', 'dl_early_accel_report_100plays.md', 'dl_early_accel_predraft_report_100plays.md', 'dl_attempt_specific_transfer_report_100plays.md', 'dl_40_accel_football_report_100plays.md', 'dl_transfer_model_report.md', 'dl_pressure_context_report.md', 'dl_game_speed_ablation_report_100plays.md', 'dl_early_accel_score_audit.md', 'game_speed_by_position_report_100plays.md', 'position_specific_game_speed_report_100plays.md', 'dl_target_uncertainty_report_100plays.md', 'dl_fixed_window_game_speed_report.md', 'dl_contextual_game_speed_report.md', 'speed_analysis_report.md']:
    report = OUTPUT / name
    if report.exists():
        print(f'\n\n## {name}\n')
        print(report.read_text(encoding='utf-8'))
svg = OUTPUT / 'combine_40_repeatability.svg'
if svg.exists(): display(SVG(filename=str(svg)))
explorer = OUTPUT / 'speed_profile_explorer.html'
if explorer.exists():
    print(f'\nProspect explorer generated at: {explorer}')

for name in ['position_phase_nested_selection_100plays.json', 'dl_attempt_specific_transfer_results_100plays.json', 'dl_game_speed_target_precision_100plays.json', 'dl_target_uncertainty_results_100plays.json', 'dl_fixed_window_game_speed_results.json', 'dl_contextual_game_speed_results.json']:
    report = OUTPUT / name
    if report.exists():
        print(f'\n\n## {name}\n')
        print(report.read_text(encoding='utf-8'))
figure = OUTPUT / 'dl_early_acceleration_summary.svg'
if figure.exists(): display(SVG(filename=str(figure)))
trace_figure = OUTPUT / 'forty_raw_trace_review.svg'
if trace_figure.exists(): display(SVG(filename=str(trace_figure)))
